# Assignment 2B — RAG Pipeline
*Chunking, dense / keyword / hybrid search, reranking, grounded generation and an LLM judge*

**Domain:** Medical and clinical literature (Type 2 Diabetes management)

**Generator:** Assignment 1 model — `mistralai/Mistral-7B-v0.1` + CPT checkpoint + QLoRA adapter

| Part | What we do | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answers and an LLM judge | 7 |
| **Total** | | **20** |

**Our choice:** the final run is on the lab server, where the Assignment 1 model files are. The brief asks for one notebook for the whole assignment, with all outputs visible.

**Models used, and where each choice comes from**

| Role | Model | Source |
| --- | --- | --- |
| Generator (fixed) | `mistralai/Mistral-7B-v0.1` + CPT + QLoRA adapter | 2B brief, Overview and D1.2: reuse the Assignment 1 model. We do not switch it (our choice) |
| Embedding | `BAAI/bge-base-en-v1.5` | Our choice. The brief's example `all-MiniLM-L6-v2` stops at 256 tokens; A2 allows switching to a model with a longer limit |
| Vector index | FAISS `IndexFlatIP` | 2B brief, A1.1 |
| Keyword search | BM25 (**our choice:** `rank-bm25`) | 2B brief, B2 asks for a BM25 index. The library is our choice |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-6-v2` | 2B brief, C1 example |
| Judge | `Qwen/Qwen2.5-7B-Instruct` (option: Llama 3.1 8B Instruct, gated) | 2B brief, D2: different from the generator, "Llama 3.1 8B Instruct or Qwen2.5 7B Instruct". Our choice: Qwen differs from Mistral |

**Setup (Assignment 1 conventions, our choice, not in the brief).** This cell follows the Assignment 1 notebook: the same installs, the same device check, and the same Hugging Face cache. It is needed on the Kubeflow lab server.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: finds out where the notebook is running (Kubeflow, Colab or local),
# picks a folder to store the files, sets the Hugging Face cache folder and checks the GPU.
# Why it runs first: Hugging Face reads HF_HOME when transformers is imported, so it must be set before.
# Output: environment, device, dtype, GPU name and the folders used.
# ---------------------------------------------------------------------------
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:      # no "google" package at all, so this is not Colab
        pass
    return "local"


def setup():
    """Find the storage root, point the Hugging Face cache at it, and detect the GPU and dtype."""
    env = detect_env()
    # Storage root per platform (Colab's disk is wiped when the session ends).
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    for path in candidates:          # first folder we can write to wins
        try:
            os.makedirs(path, exist_ok=True)
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # Model cache inside the root. Set before transformers is imported.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BAR"] = "1"   # no download bars (they show up empty in the exported notebook)

    # Optional Hugging Face token: use it if it is set (environment variable or Colab secret).
    # This model is public, so the notebook also works without one.
    token = os.environ.get("HF_TOKEN")
    if not token and env == "colab":
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token

    # Device and dtype: GPU if there is one, bf16 when the GPU supports it, else fp16, else fp32 on CPU.
    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(ENV=env, ROOT=root, CACHE_ROOT=cache_root, DEVICE=device,
                            DTYPE=dtype, IS_GPU=(device == "cuda"), GPU_NAME=gpu_name, GPU_MEM_GB=gpu_mem)
    print(f"env={env}  device={device}  dtype={dtype}  gpu={gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"storage root : {root}")
    print(f"HF cache     : {cache_root}")
    print(f"HF token     : {'found' if token else 'not set (needed only for gated models such as Llama 3.1)'}")
    return state


L = setup()

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the input and output paths, checks that every input exists and fixes the seed.
# Why: every later step reads these paths, and a missing file should fail here, not halfway through.
# Output: OK or MISSING for each input.
# ---------------------------------------------------------------------------

import json
import random
import re
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Kubeflow lab layout: this notebook is /home/jovyan/assessment/Ass-2/Assignment2B.ipynb, so ../Ass-1 is
# /home/jovyan/assessment/Ass-1 (shown as /assessment/Ass-1 in the Jupyter file browser). The setup cell puts the
# Hugging Face cache in the same storage root, at /home/jovyan/assessment/hf-cache.
ASS1 = Path("../Ass-1").resolve()
CORPUS_DIR = ASS1 / "domain_corpus"                      # 14 cleaned .txt files
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"        # 400 rows; the LAST 80 are the eval set
# Do NOT use ASS1 / "instruction_eval.jsonl": it holds 93 pairs, none of them in instruction_dataset.jsonl.
# The Assignment 1 B1 output says the eval split has 80 pairs (the last 80 rows of the dataset).

# Our choice, from Assignment 1 conventions (cells 29 and 49): model files are under ASS1 / "persistent".
# The prints below show where they resolve.
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"
ADAPTER_DIR = ASS1 / "persistent" / "adapter"
print(f"Assignment 1 folder : {ASS1}")
print(f"CPT checkpoint      : {CPT_DIR}")
print(f"QLoRA adapter       : {ADAPTER_DIR}")
print(f"HF cache            : {L.CACHE_ROOT}")

DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)


def count_rows(path):
    """Number of non-empty lines in a JSONL file."""
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


checks = [
    ("corpus: 14 .txt files", CORPUS_DIR.is_dir() and len(list(CORPUS_DIR.glob("*.txt"))) == 14),
    ("instruction_dataset.jsonl: 400 rows", DATASET_PATH.exists() and count_rows(DATASET_PATH) == 400),
    ("CPT checkpoint folder (lab server only)", CPT_DIR.is_dir()),
    ("QLoRA adapter folder (lab server only)", ADAPTER_DIR.is_dir()),
]
for label, ok in checks:
    print(f"{'OK' if ok else 'MISSING':8} {label}")


## Prerequisites — Environment Setup

We install the packages that are missing and print their versions, so anyone can see which versions gave these results.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: installs the packages that are missing.
# Why: the notebook should run on a fresh lab server or Colab runtime.
# Output: which packages were already there and which were installed.
# ---------------------------------------------------------------------------

import importlib.util
import subprocess
import sys

# pip name -> import name
PACKAGES = {
    "sentence-transformers": "sentence_transformers",  # embeddings (A, B) and CrossEncoder (C)
    "faiss-cpu": "faiss",                              # IndexFlatIP vector index (A, B)
    "rank-bm25": "rank_bm25",                          # BM25Okapi keyword index (B)
    "transformers": "transformers",                    # tokenizers, generator and judge (A, D)
    "peft": "peft",                                    # QLoRA adapter (D)
    "bitsandbytes": "bitsandbytes",                    # 4-bit loading (D)
    "accelerate": "accelerate",                        # device_map loading (D)
    "pandas": "pandas",                                # tables
    "nbconvert": "nbconvert",                          # HTML export (submission)
}

missing = [pip for pip, module in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all packages were already installed")
print("already there:", ", ".join(p for p in PACKAGES if p not in missing))


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines print_table(), a small helper that prints aligned tables.
# All reports in this notebook use it. Column widths are calculated from the data,
# so the columns stay aligned even when a value is long.
# ---------------------------------------------------------------------------
def print_table(headers, rows, title=None, align=None, total_row=None, save_as=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples, cells already formatted as strings so the caller
                controls thousands separators and dashes for values that don't apply
    title     : optional heading printed above a rule
    align     : per-column "l"/"r"; defaults to label-left, numbers-right
    total_row : optional summary row, printed below a separating rule
    save_as   : optional file name; the table is also saved as outputs/<save_as>.csv
    """
    # Default alignment: first column left, the rest right.
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    # Include the total row when working out widths so it fits too.
    body = list(rows) + ([total_row] if total_row else [])
    # Width of each column = longest cell in that column, header included.
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "  # space between columns
    # Full table width, used for the ruled lines.
    width = sum(widths) + len(gap) * (len(widths) - 1)

    # Format one row: pad each cell to its column width.
    def line(cells):
        """Format one table row: pad each cell to its column width."""
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    # Print the title, the header, the rows and the optional total row.
    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    if total_row:
        print("-" * width)
        print(line(total_row))
    print("=" * width)

    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the Python version, the package versions, torch and the GPU.
# Why: so the output shows which versions and which GPU gave these results.
# Output: a versions table.
# ---------------------------------------------------------------------------

import platform
from importlib import metadata

import torch

rows = [("Python", platform.python_version()), ("torch", torch.__version__)]
for pip_name in PACKAGES:
    try:
        rows.append((pip_name, metadata.version(pip_name)))
    except metadata.PackageNotFoundError:
        rows.append((pip_name, "NOT INSTALLED"))
rows.append(("GPU", f"{L.GPU_NAME or 'none'} ({L.GPU_MEM_GB} GB)"))
rows.append(("dtype", L.DTYPE))
print_table(["Item", "Version or value"], rows, title="Versions", save_as="versions")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines md_safe() and working, copied from Assignment 1.
# Why: md_safe prints model answers safely in markdown; working shows progress during slow model loads.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------
# Helper for markdown tables: puts the model text on one line and escapes the characters that markdown uses for formatting.
def md_safe(text, limit=400):
    """Return model output as one line of text that renders as plain text in markdown."""
    t = " ".join(str(text).split())
    if len(t) > limit:
        t = t[:limit].rstrip() + " ..."
    for ch in "\\`*_[]<>!|#":
        t = t.replace(ch, "\\" + ch)
    return t


# Helper that prints a live status line (time and folder size) for slow steps like loading a 14 GB model,
# so we can see that it is still running.
import threading
import time as _time_mod
from pathlib import Path as _Path


def _folder_gb(folder):
    """Size of everything inside a folder in GB (0 if it does not exist yet)."""
    try:
        return sum(f.stat().st_size for f in _Path(folder).rglob("*") if f.is_file() and not f.is_symlink()) / 1024**3
    except OSError:
        return 0.0


class working:
    """Context manager: shows 'label ... 2m10s' (and the folder size, if given) while the code inside runs."""

    def __init__(self, label, folder=None, expected_gb=None):
        """Remember the message and, optionally, the folder to measure and its expected final size."""
        self.label, self.folder, self.expected_gb = label, folder, expected_gb

    def _line(self):
        """Build the status text for the current moment."""
        secs = int(_time_mod.time() - self.t0)
        text = f"{self.label} ... {secs // 60}m{secs % 60:02d}s"
        if self.folder is not None:
            gb = _folder_gb(self.folder)
            text += f"  ({gb:.1f} of ~{self.expected_gb:.1f} GB)" if self.expected_gb else f"  ({gb:.1f} GB so far)"
        return text

    def _run(self):
        """Background loop: rewrite the status line every 5 seconds until the step ends."""
        while not self._stop.wait(5):
            print("\r" + self._line() + "   ", end="", flush=True)

    def __enter__(self):
        """Start the status line."""
        self.t0, self._stop = _time_mod.time(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        """Stop the status line and print the final time."""
        self._stop.set()
        self._thread.join()
        print("\r" + self._line().replace(" ...", " ... done in", 1) + "   ")
        return False

## Before Part A — 10 queries and their answer spans

**Goal:** fix the 10 queries, their expected answers and their answer spans. Parts A, B and C score every search with these spans.

**What we do (our choice, not in the brief):**
1. Take the Assignment 1 eval set: the last 80 rows of `instruction_dataset.jsonl`. The dataset has no passage field, so each row is matched to the corpus.
2. For each row, find the corpus sentence (8 to 40 words) with the most word overlap with the expected answer. That sentence is the span, and its file is the passage.
3. Keep spans that occur exactly once in the corpus. Keep the best row per file and take 10 rows.
4. The expected answer is the row's `response`. The judge in D2 uses it.
5. Human check: read each span against its query, and replace any span that does not answer it.

A retrieved chunk is a **hit** if it contains the answer span.


**Our choice: linking each eval question to its passage.** The brief does not give a question-to-passage link, and `instruction_dataset.jsonl` has only `instruction` and `response`. So we match each eval answer to the corpus ourselves: the corpus sentence with the most word overlap with the answer is the span, and its file is the passage. The course may not accept this method.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the last 80 eval rows, matches each answer to a corpus sentence and picks 10 queries, one per file.
# Why: the brief needs one answer span per query. The match is our choice (see the note above).
# Output: the 10 queries with their passage file and span.
# ---------------------------------------------------------------------------

eval_rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8")][-80:]
assert len(eval_rows) == 80

STOP = set("the a an of and or to in on for with by is are was were be as at from that this it its which".split())

def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

# Candidate spans: corpus sentences of 8 to 40 words, with their file.
SENTENCES = []
for f in sorted(CORPUS_DIR.glob("*.txt")):
    text = " ".join(f.read_text(encoding="utf-8").split())
    for sent in re.split(r"(?<=[.!?])\s+", text):
        if 8 <= len(sent.split()) <= 40:
            SENTENCES.append((f.name, sent))

def best_span(response):
    """Score = overlap / sqrt(sentence length). Ties go to the shorter sentence."""
    want = tokens(response)
    scored = []
    for source, sent in SENTENCES:
        have = tokens(sent)
        if have:
            overlap = len(want & have)
            scored.append((overlap / len(have) ** 0.5, overlap, -len(sent), source, sent))
    return max(scored)

matches = []
for row in eval_rows:
    score, overlap, _, source, sent = best_span(row["response"])
    matches.append({"overlap": overlap, "source": source, "span": sent,
                    "query": row["instruction"], "expected_answer": row["response"]})

# Keep only spans that occur exactly once in the whole corpus, then the best match per file.
def norm(t): return " ".join(t.lower().split())
CORPUS_NORM = norm(" ".join(f.read_text(encoding="utf-8") for f in CORPUS_DIR.glob("*.txt")))
unique = [m for m in sorted(matches, key=lambda m: -m["overlap"]) if CORPUS_NORM.count(norm(m["span"])) == 1]
best_per_file = {}
for m in unique:
    best_per_file.setdefault(m["source"], m)
QUERIES = []
for qid, m in enumerate(sorted(best_per_file.values(), key=lambda m: -m["overlap"])[:10], start=1):
    QUERIES.append({"qid": f"Q{qid}", "source_file": m["source"], "query": m["query"],
                    "expected_answer": m["expected_answer"], "span": m["span"]})

assert len(QUERIES) == 10 and len({q["source_file"] for q in QUERIES}) == 10

**Human check (our choice, not in the brief).** Word overlap can pick a sentence that shares words without answering the question. Each span below was read against its query. Q9 and Q10 matched the wrong file: their answers are lines of the "Lifestyle Goals in Diabetes" list in the ICMR guideline, so those lines are their spans. Read the 10 spans again before the final run, and change `HUMAN_CHECK` if you disagree.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: applies the human check: "ok" keeps the span, a sentence replaces it.
# Why: word overlap can pick a sentence that shares words without answering the question.
# Output: the 10 queries with their final source file and span.
# ---------------------------------------------------------------------------

HUMAN_CHECK = {
    "Q1": "ok", "Q2": "ok", "Q3": "ok", "Q4": "ok", "Q5": "ok", "Q6": "ok", "Q7": "ok", "Q8": "ok",
    "Q9": "Advise other behavioural changes for: smoking, other tobacco products and alcohol",
    "Q10": "To individualise the diet according to complications and co-morbidities",
}

for qid, value in HUMAN_CHECK.items():
    if value != "ok":
        q = next(x for x in QUERIES if x["qid"] == qid)
        assert CORPUS_NORM.count(norm(value)) == 1, f"{qid}: the replacement must occur exactly once in the corpus"
        q["span"] = value
        q["source_file"] = next(f.name for f in CORPUS_DIR.glob("*.txt") if norm(value) in norm(f.read_text(encoding="utf-8")))

assert set(HUMAN_CHECK) == {q["qid"] for q in QUERIES}, "every query needs a human check"
for q in QUERIES:
    print(f"{q['qid']}  ({q['source_file']})\n  query: {q['query']}\n  span : {q['span']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: saves the 10 queries and shows them.
# Why: data/answer_spans.json is what the hit rates in Parts A, B and C use.
# Output: data/answer_spans.json and a table of the 10 queries.
# ---------------------------------------------------------------------------

def is_hit(chunk_text, span):
    """True if the chunk contains the answer span."""
    return norm(span) in norm(chunk_text)

(DATA_DIR / "answer_spans.json").write_text(json.dumps(QUERIES, indent=1, ensure_ascii=False), encoding="utf-8")
print_table(["qid", "source file", "query", "expected answer (first 60 chars)", "span (first 60 chars)"],
            [(q["qid"], q["source_file"], q["query"], q["expected_answer"][:60], q["span"][:60]) for q in QUERIES],
            title="The 10 queries", save_as="query_table")

## Part A — Dense search and chunk size (5 marks)

**Goal:** build dense search with FAISS and choose the chunk size for Parts B to D.

**Embedding model.** `all-MiniLM-L6-v2` cuts input at 256 tokens, so 400-token chunks would be truncated. We switch to `BAAI/bge-base-en-v1.5` (512-token limit), so both sizes fit. Tokens are counted with this model's tokenizer.

**What we do:**
1. A1.1 — one reusable dense search function (FAISS `IndexFlatIP`).
2. A2.1 — chunk the corpus at 100 and 400 tokens, run the 10 queries, fill the table.
3. A2.2 — choose the chunk size.
4. A1.2 — report the setup for the final chunk size.

*Note on the brief:* the A2 table header says 500, but the A2 text and the submission checklist say 400, so we run 100 and 400.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the embedding model and prints its limit and vector size.
# Why: the brief asks us to note the model's maximum input length and count tokens with its tokenizer.
# Output: model name, max_seq_length, vector size.
# ---------------------------------------------------------------------------

from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"   # our choice: 512-token limit (the brief's example all-MiniLM-L6-v2 stops at 256; A2 allows switching)
EMBED_BATCH = 64
# Our choice: bge v1.5 is trained to read a query prefix. We add it to queries only, never to chunks.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

with working("Loading the embedding model"):
    embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE)
embed_tok = embed_model.tokenizer          # counts the chunk tokens for A2
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"model              : {EMBED_MODEL_ID}")
print(f"max_seq_length     : {embed_model.max_seq_length} tokens")
print(f"vector size        : {EMBED_DIM}")


### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines dense_search(), the one reusable dense search function.
# Why: Parts A, B and C all call it, so every method is searched the same way.
# Output: nothing (definition only). The index build time is printed in A1.2.
# ---------------------------------------------------------------------------

import time

import faiss

_INDEX_CACHE = {}   # (id(chunks), len(chunks)) -> (chunks, FAISS index, build seconds)


def dense_search(chunks, queries, k):
    """Embed the chunks once (cached), build a FAISS IndexFlatIP, and return the top k for each query.

    Returns one list per query. Each item is a dict with rank (from 1), chunk_id, score and text.
    """
    key = (id(chunks), len(chunks))
    if key not in _INDEX_CACHE:
        t0 = time.perf_counter()
        vectors = embed_model.encode([c["text"] for c in chunks], batch_size=EMBED_BATCH,
                                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        _INDEX_CACHE[key] = (chunks, index, time.perf_counter() - t0)
    _, index, _ = _INDEX_CACHE[key]

    query_vectors = embed_model.encode([QUERY_PREFIX + q for q in queries], batch_size=EMBED_BATCH,
                                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(query_vectors, k)
    results = []
    for qi in range(len(queries)):
        results.append([{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(s), "text": chunks[j]["text"]}
                        for r, (s, j) in enumerate(zip(scores[qi], ids[qi])) if j >= 0])
    return results


def index_build_seconds(chunks):
    """Seconds spent encoding the chunks and calling index.add for this chunk set."""
    if (id(chunks), len(chunks)) not in _INDEX_CACHE:
        dense_search(chunks, [QUERIES[0]["query"]], 1)
    return _INDEX_CACHE[(id(chunks), len(chunks))][2]


### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over the 14 documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines chunk_corpus(), fixed-size token chunking with no overlap.
# Why: the brief asks for 100- and 400-token chunks counted with the embedding model's tokenizer.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def chunk_corpus(size):
    """Split each .txt file into chunks of `size` tokens with no overlap.

    Tokens are counted with the embedding tokenizer. Each chunk's text is cut from the original
    file with the token offsets, so the wording is not lowercased.
    """
    chunks = []
    for f in sorted(CORPUS_DIR.glob("*.txt")):
        text = f.read_text(encoding="utf-8")
        offsets = embed_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        for start in range(0, len(offsets), size):
            part = offsets[start:start + size]
            chunks.append({"chunk_id": f"{f.stem}#{start // size}", "doc": f.name,
                           "text": text[part[0][0]:part[-1][1]], "n_tokens": len(part)})
    return chunks


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search on the 10 queries for each chunk size and fills the A2 table.
# Why: this is the evidence for choosing the chunk size.
# Output: the A2 table (chunks, hit rate@3, tokens sent per query).
# ---------------------------------------------------------------------------

A2_SIZES = [100, 400]       # the sizes in the A2 text and the submission checklist
A2_HIT = {}                  # chunk size -> hit rate@3 (%)
A2_TOKENS = {}               # chunk size -> tokens sent per query (top 3)
chunk_sets = {}              # chunk size -> list of chunks
a2_rows = []
for size in A2_SIZES:
    chunks_for_size = chunk_corpus(size)
    assert max(c["n_tokens"] for c in chunks_for_size) <= size
    chunk_sets[size] = chunks_for_size
    by_id = {c["chunk_id"]: c for c in chunks_for_size}
    tops = dense_search(chunks_for_size, [q["query"] for q in QUERIES], 3)
    hits = sum(any(is_hit(r["text"], q["span"]) for r in top) for q, top in zip(QUERIES, tops))
    tokens = sum(sum(by_id[r["chunk_id"]]["n_tokens"] for r in top) for top in tops) / len(QUERIES)
    A2_HIT[size] = 100 * hits / len(QUERIES)
    A2_TOKENS[size] = tokens
    a2_rows.append((str(size), f"{len(chunks_for_size):,}", f"{A2_HIT[size]:.0f}", f"{tokens:.0f}"))
print_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"],
            a2_rows, title="A2 chunk size test (top 3, 10 queries)", save_as="a2_chunk_size")

# Check: 100-token chunks should be about 4 times the 400-token count.
ratio = len(chunk_sets[100]) / len(chunk_sets[400])
print(f"100-token chunks / 400-token chunks = {ratio:.2f} (about 4 expected)")
assert 3 < ratio < 5, ratio


### A2.2 Chosen chunk size (1 mark)

*Inference cell below: two or three sentences built from the A2 numbers. Weigh hit rate, tokens per query, and how much context the Assignment 1 model can handle: its adapter was trained with `SFT_MAX_LEN = 512` (Assignment 1, cell 49).*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the chosen chunk size and measures the Part D prompt length with the Mistral tokenizer.
# Why: the generator counts tokens differently from the embedding model.
# Output: the chosen size and the prompt length in Mistral tokens.
# ---------------------------------------------------------------------------

from transformers import AutoTokenizer

# Our choice: the size with the highest hit rate@3. A tie goes to the smaller size, which sends fewer tokens.
CHUNK_SIZE = min(A2_SIZES, key=lambda s: (-A2_HIT[s], s))
chunks = chunk_sets[CHUNK_SIZE]
print(f"chosen chunk size: {CHUNK_SIZE} tokens ({len(chunks):,} chunks)")

# Our check: count one Part D prompt (3 chunks and a question) with the Mistral tokenizer from the adapter folder.
# The layout follows GEN_PROMPT in D1.1, so the count is close to the real prompt.
mistral_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
top3 = dense_search(chunks, [QUERIES[0]["query"]], 3)[0]
prompt_text = "\n".join(f"[{r['rank']}] {r['text']}" for r in top3) + f"\n\nQuestion: {QUERIES[0]['query']}"
PROMPT_TOKENS = len(mistral_tok(prompt_text).input_ids)
print(f"Mistral tokens for one prompt (3 chunks and a question): {PROMPT_TOKENS}")
print("adapter training length: 512 tokens (SFT_MAX_LEN, Assignment 1)")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - A2.2: the chosen chunk size and why (numbers come from the A2 table).
# Why: the reason must match the numbers, so the text is built from them.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

fits = "within" if PROMPT_TOKENS <= 512 else "above"
display(Markdown(
    f"**A2.2 chosen size: {CHUNK_SIZE} tokens, used in Parts B, C and D.** Hit rate@3 was {A2_HIT[100]:.0f}% at "
    f"100 tokens and {A2_HIT[400]:.0f}% at 400, and the top 3 sent {A2_TOKENS[100]:.0f} and {A2_TOKENS[400]:.0f} "
    f"tokens per query. One Part D prompt at {CHUNK_SIZE} tokens is {PROMPT_TOKENS} Mistral tokens, which is {fits} "
    f"the 512 tokens the Assignment 1 adapter was trained on. Rule used: the highest hit rate@3 wins, and a tie "
    f"goes to the smaller size, which sends fewer tokens."))


### A1.2 Dense search setup (1 mark)

Reported for the final chunk set chosen in A2.2.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the dense search setup for the final chunk set.
# Why: the brief asks for these numbers for the final chunk set.
# Output: a table with the model, vector size, similarity, batch size, chunk count and build time.
# ---------------------------------------------------------------------------

print_table(["Setting", "Value"], [
    ("Embedding model", EMBED_MODEL_ID),
    ("Vector size", str(EMBED_DIM)),
    ("Similarity measure", "inner product on L2-normalised vectors (cosine)"),
    ("Batch size", str(EMBED_BATCH)),
    ("Number of chunks", f"{len(chunks):,}"),
    ("Index build time (s)", f"{index_build_seconds(chunks):.2f}"),
], title=f"A1.2 dense search setup, chunk size {CHUNK_SIZE} tokens", save_as="a1_2_setup")


## Part B — Keyword and hybrid search (4 marks)

**Goal:** compare dense search, BM25 and a hybrid of the two on the same 10 queries, top 5 each.

**What we do:**
1. B1 — dense search from Part A.
2. B2 — BM25 index, top 5.
3. B3 — hybrid with Reciprocal Rank Fusion over the top 20 of each method:
   `RRF(d) = 1 / (60 + rank in dense) + 1 / (60 + rank in BM25)`. Ranks start at 1; a chunk missing from a list gets nothing from it.
4. B4 — which method works best, with one example query.

| Metric | How |
| --- | --- |
| Query latency (ms) | time to return the top 5 for one query, averaged over the 10 queries, after one warm-up run |
| Hit rate@1 (%) | share of queries whose first result contains the span |
| Hit rate@3 (%) | share of queries with the span in the top 3 |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the shared evaluate() helper used for every method in Parts B and C.
# Why: one helper means the rows of the table are measured the same way.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

import time


def evaluate(search_fn, k=5):
    """Run search_fn(query, k) on the 10 queries after one warm-up.

    Returns avg latency (ms), hit rate@1 (%), hit rate@3 (%) and the ranked lists per query.
    """
    search_fn(QUERIES[0]["query"], k)   # warm-up, not timed
    latencies, ranked = [], {}
    for q in QUERIES:
        t0 = time.perf_counter()
        ranked[q["qid"]] = search_fn(q["query"], k)
        latencies.append((time.perf_counter() - t0) * 1000)

    def hit_at(n):
        """Share of queries whose top n contain the answer span, in %."""
        found = sum(any(is_hit(r["text"], q["span"]) for r in ranked[q["qid"]][:n]) for q in QUERIES)
        return 100 * found / len(QUERIES)

    return {"latency_ms": sum(latencies) / len(latencies), "hit1": hit_at(1), "hit3": hit_at(3), "ranked": ranked}


### B1 Dense (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search (top 5) through evaluate().
# Why: fills the Dense row of the B table.
# Output: latency, hit rate@1, hit rate@3 for Dense.
# ---------------------------------------------------------------------------

b_results = {}   # method name -> evaluate() result
dense_fn = lambda q, k: dense_search(chunks, [q], k)[0]
b_results["Dense"] = evaluate(dense_fn)
print(f"Dense: hit rate@1 {b_results['Dense']['hit1']:.0f}%, hit rate@3 {b_results['Dense']['hit3']:.0f}%")


### B2 BM25 (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: builds a BM25 index over the chunks and runs it (top 5).
# Why: keyword search catches exact terms such as drug names and HbA1c.
# Output: latency, hit rate@1, hit rate@3 for BM25.
# ---------------------------------------------------------------------------

from rank_bm25 import BM25Okapi


def bm25_tokens(text):
    """Lowercase words and numbers. Keeps terms like hba1c and sglt2 whole."""
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])


def bm25_fn(query, k):
    """Top k chunks for one query, in the same format as dense_search."""
    scores = bm25.get_scores(bm25_tokens(query))
    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(scores[j]), "text": chunks[j]["text"]}
            for r, j in enumerate(top)]


b_results["BM25"] = evaluate(bm25_fn)
print(f"BM25: hit rate@1 {b_results['BM25']['hit1']:.0f}%, hit rate@3 {b_results['BM25']['hit3']:.0f}%")


### B3 Hybrid with RRF (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines rrf_fuse(), tests it on a toy case, and runs the hybrid search (top 5).
# Why: RRF combines the two ranked lists without needing their scores to be on the same scale.
# Output: the toy test result and the Hybrid row.
# ---------------------------------------------------------------------------


def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Reciprocal rank fusion: score = sum of 1 / (k + rank) over the lists the chunk appears in."""
    score = {}
    for ids in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ids, start=1):
            score[cid] = score.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(score.items(), key=lambda item: -item[1])[:top]


# Toy test: X is 2nd in the dense list and 3rd in the BM25 list, Y is 1st in the dense list only.
# X should beat Y.
toy = rrf_fuse(["Y", "X", "Z"], ["W", "Q", "X"], top=3)
assert toy[0][0] == "X", toy
print("RRF toy test passed: X (2nd and 3rd) beats Y (1st in one list).")

chunk_by_id = {c["chunk_id"]: c for c in chunks}


def hybrid_fn(query, k):
    """Top 20 from dense and from BM25, fused with RRF, keep the top k."""
    dense_top = dense_fn(query, 20)
    bm25_top = bm25_fn(query, 20)
    fused = rrf_fuse([r["chunk_id"] for r in dense_top], [r["chunk_id"] for r in bm25_top], top=k)
    return [{"rank": r + 1, "chunk_id": cid, "score": s, "text": chunk_by_id[cid]["text"]}
            for r, (cid, s) in enumerate(fused)]


b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)
print(f"Hybrid (RRF): hit rate@1 {b_results['Hybrid (RRF)']['hit1']:.0f}%, "
      f"hit rate@3 {b_results['Hybrid (RRF)']['hit3']:.0f}%")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the B comparison table and the rank of the answer chunk per query and method.
# Why: the table is graded; the per-query ranks show where each method found the answer.
# Output: the B table and the per-query rank table.
# ---------------------------------------------------------------------------


def rank_of(method, qid, span):
    """Rank of the first top-5 chunk that contains the span, or "-" if none does."""
    for r in b_results[method]["ranked"][qid]:
        if is_hit(r["text"], span):
            return str(r["rank"])
    return "-"


print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [(m, f"{v['latency_ms']:.1f}", f"{v['hit1']:.0f}", f"{v['hit3']:.0f}") for m, v in b_results.items()],
            title=f"Part B search comparison (chunk size {CHUNK_SIZE}, top 5)", save_as="b_comparison")
print_table(["Query"] + list(b_results),
            [(q["qid"], *[rank_of(m, q["qid"], q["span"]) for m in b_results]) for q in QUERIES],
            title="Rank of the answer chunk (- = not in the top 5)", save_as="b_ranks")


### B4 Best method (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - B4: which method works best and why, with one example query.
# Why: the text is built from the B table, so it stays in line with the numbers.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Our choice: best hit rate@1, then hit rate@3, then the faster method.
BEST_METHOD = max(b_results, key=lambda m: (b_results[m]["hit1"], b_results[m]["hit3"], -b_results[m]["latency_ms"]))
REASON = {
    "Dense": "dense search matches meaning, so it finds answers that are worded differently from the question",
    "BM25": "BM25 matches exact words such as drug names and numbers, which dense search can blur",
    "Hybrid (RRF)": "the hybrid keeps a chunk that either list ranks well, so it recovers answers that one method misses",
}
# Example: the first query where the answer chunk ranks differently across the methods (Q1 if none does).
ex = next((q for q in QUERIES if len({rank_of(m, q["qid"], q["span"]) for m in b_results}) > 1), QUERIES[0])
display(Markdown(
    f"**B4 best method: {BEST_METHOD}.** Hit rate@1 {b_results[BEST_METHOD]['hit1']:.0f}% and hit rate@3 "
    f"{b_results[BEST_METHOD]['hit3']:.0f}%, against Dense {b_results['Dense']['hit1']:.0f}% and "
    f"BM25 {b_results['BM25']['hit1']:.0f}% at rank 1. Why: {REASON[BEST_METHOD]}. Example query {ex['qid']}: "
    f"\"{ex['query']}\" The answer chunk ranks {rank_of('Dense', ex['qid'], ex['span'])} with Dense, "
    f"{rank_of('BM25', ex['qid'], ex['span'])} with BM25 and {rank_of('Hybrid (RRF)', ex['qid'], ex['span'])} "
    f"with Hybrid (RRF)."))


## Part C — Reranking (4 marks)

**Goal:** re-score the top 10 of the best Part B method with a cross-encoder and keep the top 3.

**What we do:**
1. C1 — retrieve the top 10 with the best method; re-score with `cross-encoder/ms-marco-MiniLM-L-6-v2`; keep the top 3.
2. C2 — show the top 3 before and after reranking for each query, and the rank change rate.
3. C3 — reranking time per query and hit rate@1 before and after.
4. C4 — did reranking help?

**Rank change rate** = share of the 10 queries whose first result changes after reranking.

### C1 Rerank (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the cross-encoder and reranks the top 10 of the best method for each query.
# Why: a cross-encoder reads the query and the chunk together, so it can order them better.
# Output: the device used and final_chunks (top 3 per query).
# ---------------------------------------------------------------------------

import time

import torch
from sentence_transformers import CrossEncoder

RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # 2B brief, C1 example
with working("Loading the cross-encoder"):
    ce = CrossEncoder(RERANK_MODEL_ID, max_length=512, device=L.DEVICE)
print(f"cross-encoder: {RERANK_MODEL_ID} on {L.DEVICE}")

best_fn = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}[BEST_METHOD]


def rerank_top3(query):
    """Top 10 with the best method, reranked. Returns (before top 3, after top 3, predict seconds)."""
    top10 = best_fn(query, 10)
    pairs = [(query, c["text"]) for c in top10]
    t0 = time.perf_counter()
    scores = ce.predict(pairs)
    if L.IS_GPU:
        torch.cuda.synchronize()    # GPU work is asynchronous, so wait before stopping the timer
    seconds = time.perf_counter() - t0
    order = sorted(range(len(top10)), key=lambda i: -scores[i])[:3]
    after = [dict(top10[i], rank=r + 1, ce_score=float(scores[i])) for r, i in enumerate(order)]
    return top10[:3], after, seconds


ce.predict([(QUERIES[0]["query"], "warm-up")])   # warm-up, not timed
final_chunks, before_top3, rerank_ms = {}, {}, []
for q in QUERIES:
    before, after, seconds = rerank_top3(q["query"])
    before_top3[q["qid"]] = before
    final_chunks[q["qid"]] = after
    rerank_ms.append(seconds * 1000)
print(f"reranked {len(QUERIES)} queries; kept the top 3 of 10 for each")


### C2 Compare rankings (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the top 3 before and after reranking for each query.
# Why: the brief asks to see the change for every query.
# Output: the before and after top 3 per query, and the rank change rate.
# ---------------------------------------------------------------------------

changed = 0
for q in QUERIES:
    qid = q["qid"]
    print(f"\n{qid}  {q['query']}")
    print("  Before reranking (best method's own top 3):")
    for r in before_top3[qid]:
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {r['text'][:80]}")
    print("  After reranking (cross-encoder top 3):")
    for r in final_chunks[qid]:
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {r['text'][:80]}")
    if before_top3[qid][0]["chunk_id"] != final_chunks[qid][0]["chunk_id"]:
        changed += 1

RANK_CHANGE = 100 * changed / len(QUERIES)   # % of queries whose first result changed
print(f"\nRank change rate: {RANK_CHANGE:.0f}% of queries (first result changed)")


### C3 Measure (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: fills the reranking table.
# Why: weighs accuracy against the time reranking adds.
# Output: reranking time per query, rank change rate, hit rate@1 before and after.
# ---------------------------------------------------------------------------

hit1_before = b_results[BEST_METHOD]["hit1"]
hit1_after = 100 * sum(is_hit(final_chunks[q["qid"]][0]["text"], q["span"]) for q in QUERIES) / len(QUERIES)
RERANK_MS = sum(rerank_ms) / len(rerank_ms)
print_table(["Measure", "Value"], [
    ("Reranking time per query (ms)", f"{RERANK_MS:.1f}"),
    ("Rank change rate (%)", f"{RANK_CHANGE:.0f}"),
    ("Hit rate@1 before reranking (from Part B, same method)", f"{hit1_before:.0f}"),
    ("Hit rate@1 after reranking", f"{hit1_after:.0f}"),
], title="Part C reranking measures", save_as="c_measures")


### C4 Did reranking help? (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - C4: whether reranking helped, from the hit rates and the time added.
# Why: the text is built from the C3 numbers, so it stays in line with them.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

if hit1_after > hit1_before:
    verdict = "reranking helped"
elif hit1_after == hit1_before:
    verdict = "reranking made no difference"
else:
    verdict = "reranking lowered hit rate@1"
note = ("" if hit1_after > hit1_before else
        " The cross-encoder was trained on web search (MS MARCO), not clinical text, which may explain this.")
display(Markdown(
    f"**C4.** Hit rate@1 went from {hit1_before:.0f}% to {hit1_after:.0f}% for +{RERANK_MS:.0f} ms per query, "
    f"so {verdict}.{note}"))


## Part D — Generate and check answers (7 marks)

### D1 Generate grounded answers (3 marks)

**Goal:** answer the 10 queries with the Assignment 1 model, using only the 3 reranked chunks, with citations and a "Not found" reply.

**What we do:**
The brief has three D1 steps, one mark each:
1. D1.1 — the prompt (1 mark): answer only from the chunks, cite them, and say "Not found in the documents".
2. D1.2 — all 10 answers (1 mark), each from the 3 reranked chunks.
3. D1.3 — one query whose answer is not in the corpus (1 mark).

**Generator:** the Assignment 1 model. The brief allows an instruct model if it ignores the chunks or never says "Not found". If the run shows that, switch, name the model, and show one example of the problem.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the Assignment 1 generator: CPT model in 4-bit plus the QLoRA adapter. Settings are our choice, from Assignment 1 cells 50 and 51.
# Why: the brief asks us to reuse the Assignment 1 model as the answer generator.
# Output: the generator name and the loaded model.
# ---------------------------------------------------------------------------

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_BASE_ID = "mistralai/Mistral-7B-v0.1"   # Assignment 1 MODEL_ID (cell 18); the 2B brief says reuse the Assignment 1 model
GENERATOR_NAME = f"{GEN_BASE_ID} + CPT + QLoRA adapter (Assignment 1)"
GEN_MAX_NEW_TOKENS = 120                    # Our choice, from Assignment 1 SFT_MAX_NEW_TOKENS (cell 51): answers are 1-3 sentences
GEN_STOPS = ["Instruction:", "Question:"]   # Our choice: the next prompt starts here, so the answer ends here

assert L.IS_GPU, "4-bit loading needs a GPU. Run this notebook on the lab server."
COMPUTE_DTYPE = getattr(torch, L.DTYPE) if L.DTYPE in ("bfloat16", "float16") else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)

with working("Loading the generator (CPT model in 4-bit, then the adapter)", folder=CPT_DIR):
    gen_base = AutoModelForCausalLM.from_pretrained(CPT_DIR, quantization_config=bnb_config, device_map={"": 0})
    gen_model = PeftModel.from_pretrained(gen_base, ADAPTER_DIR).eval()
gen_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)   # saved with the chat template in Assignment 1 (cells 49 and 50)
print(f"generator: {GENERATOR_NAME}")


#### D1.1 Generation prompt (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the generation prompt and prints it.
# Why: the brief asks to show the prompt.
# Output: the prompt template.
# ---------------------------------------------------------------------------

GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Cite the chunk number you used after each fact, for example [2]. Use only the numbers 1, 2 and 3.
Stop after your answer. Do not write a new question or a new instruction.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""

print(GEN_PROMPT)


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines generate_answer(), clean_answer() and the citation and Not found checks.
# Why: the same code answers the 10 queries and the out-of-corpus query.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------

NOT_FOUND = "Not found in the documents"
CITE = re.compile(r"\[(\d+)\]")


def clean_answer(raw):
    """Cut at the first stop string, drop a trailing heading or bold block, and trim spaces."""
    text = raw
    for stop in GEN_STOPS:
        text = text.split(stop)[0]
    kept = []
    for line in text.splitlines():
        if line.lstrip().startswith(("#", "**")):
            break
        kept.append(line)
    return "\n".join(kept).strip()


def generate_answer(question, chunk_texts):
    """Greedy answer in the Assignment 1 format, from the 3 numbered chunks."""
    content = GEN_PROMPT.format(chunk_1=chunk_texts[0], chunk_2=chunk_texts[1], chunk_3=chunk_texts[2], question=question)
    prompt = gen_tok.apply_chat_template([{"role": "user", "content": content}],
                                         tokenize=False, add_generation_prompt=True).rstrip()
    ids = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        gen = gen_model.generate(**ids, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                 repetition_penalty=1.1, pad_token_id=gen_tok.eos_token_id)
    raw = gen_tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    # Same handling as Assignment 1 cell 51: keep the text after "Response:" if the model repeats it.
    if "Response:" in raw:
        raw = raw.split("Response:")[1]
    return clean_answer(raw)


def is_not_found(answer):
    """True if the reply is the Not found sentence (a final full stop is allowed)."""
    return answer.strip().rstrip(".").strip() == NOT_FOUND


def has_valid_citation(answer):
    """True if the answer cites at least one of [1], [2], [3] and no other number."""
    nums = [int(n) for n in CITE.findall(answer)]
    return bool(nums) and all(1 <= n <= 3 for n in nums)


#### D1.2 All 10 answers (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers the 10 queries with their 3 reranked chunks and shows every answer.
# Why: the brief asks for all 10 answers. The judge in D2 checks them.
# Output: a table per query (chunks cited) and the full answers.
# ---------------------------------------------------------------------------

answers = {}   # qid -> answer
for q in QUERIES:
    chunk_texts = [c["text"] for c in final_chunks[q["qid"]]]
    answers[q["qid"]] = generate_answer(q["query"], chunk_texts)

rows = []
for q in QUERIES:
    qid = q["qid"]
    cited = sorted(set(CITE.findall(answers[qid])))
    rows.append((qid, answers[qid][:80].replace("\n", " "), ",".join(cited) or "-"))
print_table(["Query", "Answer (first 80 chars)", "Chunks cited"], rows, title="D1.2 answers", save_as="d1_answers")
for q in QUERIES:
    print(f"\n{q['qid']}  {q['query']}\n  Answer: {answers[q['qid']]}")


#### D1.3 A query whose answer is not in the corpus (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs one out-of-corpus question through the same retrieval, reranking and prompt.
# Why: shows whether the model replies Not found in the documents instead of guessing.
# Output: the question, its 3 chunks, and the reply.
# ---------------------------------------------------------------------------

OOC_QUERY = "Who won the 2018 FIFA World Cup?"   # Our choice: a question the 14 diabetes documents cannot answer
_, ooc_top3, _ = rerank_top3(OOC_QUERY)          # same retrieval (best method) and reranking as Part C
for i, c in enumerate(ooc_top3, 1):
    print(f"[{i}] {c['chunk_id']}: {c['text'][:120]}")
ooc_answer = generate_answer(OOC_QUERY, [c["text"] for c in ooc_top3])
print(f"\nQuestion: {OOC_QUERY}")
print(f"Reply: {ooc_answer}")
print(f"Replied Not found: {'yes' if is_not_found(ooc_answer) else 'no'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - D1: the generator, how many answers cite the chunks, and the Not found result.
# Why: the text is built from the D1 answers, so it stays in line with them.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

n_cited = sum(has_valid_citation(answers[q["qid"]]) for q in QUERIES)
n_not_found = sum(is_not_found(answers[q["qid"]]) for q in QUERIES)
display(Markdown(
    f"**D1 result.** Generator: {GENERATOR_NAME}. With the top 3 reranked chunks, {n_cited} of 10 answers cite "
    f"only chunks [1] to [3], and {n_not_found} of 10 reply Not found. The out-of-corpus query "
    f"{'got' if is_not_found(ooc_answer) else 'did not get'} Not found in the documents."))


### D2 Check the answers with an LLM judge (4 marks)

**Goal:** grade each answer for correctness against the expected answer and for support by its 3 chunks.

**Judge model:** different from the generator. The generator is Mistral, so the judge is `Qwen/Qwen2.5-7B-Instruct`. `meta-llama/Llama-3.1-8B-Instruct` is an option only (gated, needs `HF_TOKEN`). Greedy decoding (temperature 0).

**What we do:**
1. D2.1 — run the judge on the 10 answers; per-query table and summary counts.
2. D2.2 — spot-check 3 verdicts by hand.
3. D2.3 — find the cause of one wrong or unsupported answer.
4. D2.4 — suggest one fix.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: frees the generator and loads the judge model.
# Why: two 7B models do not need to share the GPU at the same time.
# Output: the judge name. The generator is freed before the judge loads.
# ---------------------------------------------------------------------------

import gc

from transformers import AutoTokenizer

del gen_model, gen_base
gc.collect()
torch.cuda.empty_cache()

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # 2B brief, D2 example. The generator is Mistral, so Qwen differs.
assert JUDGE_MODEL_ID not in GENERATOR_NAME, "the judge must be a different model from the generator"
judge_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
with working("Loading the judge"):
    judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, quantization_config=judge_bnb,
                                                       device_map={"": 0}).eval()


def judge_generate(prompt_text):
    """Greedy judge reply (temperature 0), using the judge's own chat template."""
    text = judge_tok.apply_chat_template([{"role": "user", "content": prompt_text}],
                                         tokenize=False, add_generation_prompt=True)
    ids = judge_tok(text, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**ids, max_new_tokens=120, do_sample=False, pad_token_id=judge_tok.eos_token_id)
    return judge_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


print(f"judge: {JUDGE_MODEL_ID}")


#### D2.1 Judge prompt and results (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the judge prompt and prints it.
# Why: the brief asks to show the prompt; it follows every hint in the brief.
# Output: the judge prompt.
# ---------------------------------------------------------------------------

JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, or says "Not found in the documents".
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, quote the unsupported statement.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs the judge on the 10 answers and parses the verdicts.
# Why: a fixed output format lets us read the verdicts with code.
# Output: the per-query table and the summary counts.
# ---------------------------------------------------------------------------


def parse_verdict(text):
    """Read the CORRECT, SUPPORTED and REASON lines. A line that is not in the format is UNPARSED."""
    def pick(label):
        m = re.search(rf"^{label}:\s*(.+)$", text, re.M)
        return m.group(1).strip() if m else ""
    correct = pick("CORRECT")
    supported = pick("SUPPORTED")
    return {"correct": correct if correct in ("CORRECT", "PARTLY CORRECT", "WRONG") else "UNPARSED",
            "supported": supported if supported in ("YES", "NO") else "UNPARSED",
            "reason": pick("REASON")}


verdicts = {}
for q in QUERIES:
    qid = q["qid"]
    texts = [c["text"] for c in final_chunks[qid]]
    prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                                 chunk_2=texts[1], chunk_3=texts[2], answer=answers[qid])
    raw = judge_generate(prompt)
    verdicts[qid] = dict(parse_verdict(raw), raw=raw)

unparsed = [qid for qid, v in verdicts.items() if "UNPARSED" in (v["correct"], v["supported"])]
query_text = {q["qid"]: q["query"] for q in QUERIES}
print_table(["Query", "Correct", "Supported", "Reason"],
            [(f"{qid}: {query_text[qid]}", v["correct"], v["supported"], v["reason"]) for qid, v in verdicts.items()],
            title="D2.1 judge results", align=["l", "l", "l", "l"], save_as="d2_verdicts")
counts = {k: sum(v["correct"] == k for v in verdicts.values()) for k in ("CORRECT", "PARTLY CORRECT", "WRONG")}
print_table(["Check", "Count out of 10"], [
    ("Correct", str(counts["CORRECT"])),
    ("Partly correct", str(counts["PARTLY CORRECT"])),
    ("Wrong", str(counts["WRONG"])),
    ("Supported by the chunks", str(sum(v["supported"] == "YES" for v in verdicts.values()))),
], title="D2.1 summary", save_as="d2_summary")
print(f"lines the judge did not follow the format: {len(unparsed)} {unparsed}")


#### D2.2 Spot-check the judge (1 mark)

*The cell below shows 3 answers (with different verdicts where possible), their 3 chunks and the judge's verdicts. Read them, then fill in the notes cell after it.*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows 3 answers with their chunks and the judge's verdicts for the manual check.
# Why: the spot-check must be done by reading the answers yourself, not by the judge.
# Output: the 3 answers, their chunks and the judge's verdicts.
# ---------------------------------------------------------------------------

# Pick 3 answers with different verdicts where possible.
spot_qids = []
for wanted in ("CORRECT", "PARTLY CORRECT", "WRONG"):
    qid = next((q["qid"] for q in QUERIES if verdicts[q["qid"]]["correct"] == wanted and q["qid"] not in spot_qids), None)
    if qid:
        spot_qids.append(qid)
for q in QUERIES:
    if len(spot_qids) >= 3:
        break
    if q["qid"] not in spot_qids:
        spot_qids.append(q["qid"])

for qid in spot_qids:
    q = next(x for x in QUERIES if x["qid"] == qid)
    print(f"\n{qid}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[qid]}")
    for i, c in enumerate(final_chunks[qid], 1):
        print(f"  [{i}] {c['text']}")
    print(f"  Judge: CORRECT={verdicts[qid]['correct']}  SUPPORTED={verdicts[qid]['supported']}  REASON={verdicts[qid]['reason']}")


**D2.2 my spot-check** *(fill in after reading the 3 answers above)*

| Query | Judge (correct / supported) | Mine (correct / supported) | Agree? |
| --- | --- | --- | --- |
| Q_ | | | |
| Q_ | | | |
| Q_ | | | |

I agree with _ of the 3 verdicts. Disagreement (if any): _one or two sentences on which verdict and why._

#### D2.3 Find the cause (1 mark)

*The cell below picks one answer marked WRONG or SUPPORTED: NO (if there is none, the first PARTLY CORRECT answer), shows its 3 chunks, and checks whether the answer span is in them: **retrieval** (the needed fact is not in the chunks) or **generation** (the fact is in a chunk, but the model did not use it or added something).*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the chosen failed answer with its 3 chunks, finds the cause and picks one fix.
# Why: is_hit on the span tells us whether retrieval or generation caused the failure, and where retrieval lost it.
# Output: the answer, its 3 chunks, the cause and the fix.
# ---------------------------------------------------------------------------


def failed(qid):
    return verdicts[qid]["correct"] == "WRONG" or verdicts[qid]["supported"] == "NO"


FAILED_QID = next((q["qid"] for q in QUERIES if failed(q["qid"])), None)
if FAILED_QID is None:
    FAILED_QID = next((q["qid"] for q in QUERIES if verdicts[q["qid"]]["correct"] == "PARTLY CORRECT"), None)
    if FAILED_QID:
        print("No answer is WRONG or unsupported. Using the weakest answer (PARTLY CORRECT).")

FAILED_CAUSE, FAILED_FIX = None, None
if FAILED_QID is None:
    print("No failed or partly correct answer to analyse.")
else:
    q = next(x for x in QUERIES if x["qid"] == FAILED_QID)
    print(f"{FAILED_QID}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[FAILED_QID]}")
    print(f"  Judge: CORRECT={verdicts[FAILED_QID]['correct']}  SUPPORTED={verdicts[FAILED_QID]['supported']}  "
          f"REASON={verdicts[FAILED_QID]['reason']}")
    for i, c in enumerate(final_chunks[FAILED_QID], 1):
        print(f"\n  [{i}] {c['chunk_id']}:\n{c['text']}")

    # Where is the answer span? In the 3 chunks, in the top 10 before reranking, in some chunk, or split.
    in_top3 = any(is_hit(c["text"], q["span"]) for c in final_chunks[FAILED_QID])
    in_top10 = any(is_hit(c["text"], q["span"]) for c in best_fn(q["query"], 10))
    in_a_chunk = any(is_hit(c["text"], q["span"]) for c in chunks)
    if in_top3:
        FAILED_CAUSE = "generation"
        FAILED_FIX = ("stricter prompt wording: tell the model to state only the fact from the chunk it cites and add "
                      "nothing else, because the fact was in the chunks but the answer did not use it correctly")
    elif in_top10:
        FAILED_CAUSE = "retrieval"
        FAILED_FIX = ("pass the top 5 reranked chunks instead of 3, because the chunk with the fact was in the top 10 "
                      "but the reranker did not put it in the top 3")
    elif in_a_chunk:
        FAILED_CAUSE = "retrieval"
        FAILED_FIX = ("rerank the top 20 instead of the top 10, because the chunk with the fact exists but the search "
                      "did not rank it high enough to reach the reranker")
    else:
        FAILED_CAUSE = "retrieval"
        FAILED_FIX = ("add a 50-token overlap between chunks, because the answer span falls across a chunk boundary, "
                      "so no single chunk holds it")
    print(f"\n  Answer span in the 3 chunks: {'yes' if in_top3 else 'no'}")
    print(f"  Cause: {FAILED_CAUSE.upper()}")


**D2.3 cause in my words** *(fill in after reading the 3 chunks above)*: _was the needed information in the chunks or not, and what did the model do with it? One or two sentences._

#### D2.4 Suggest a fix (1 mark)

*One change that matches the cause found in D2.3, and why it would help.*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - D2: judge summary, the cause and the one suggested fix.
# Why: the text is built from the D2 results, so it stays in line with them.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

supported = sum(v["supported"] == "YES" for v in verdicts.values())
display(Markdown(
    f"**D2 summary.** Judge: {JUDGE_MODEL_ID}. Correct {counts['CORRECT']}, partly correct "
    f"{counts['PARTLY CORRECT']}, wrong {counts['WRONG']} (of 10). Supported by the chunks: {supported} of 10. "
    + (f"Failed answer {FAILED_QID}: the cause is {FAILED_CAUSE}. Suggested fix: {FAILED_FIX}."
       if FAILED_CAUSE else "No failed answer to analyse.")))


## Submission

1. Restart the kernel and run all cells on the lab server.
2. Fill in the D2.2 and D2.3 notes cells from what the run shows.
3. Check that every output is visible.
4. Save the notebook (File > Save). Then, in the Ass-2 folder (the folder this notebook is in), run `jupyter nbconvert --to html Assignment2B.ipynb`. The HTML file is written next to the notebook.